# 1.install and import packges 

In [20]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-groq \
    pypdf \
    fastembed \
    chromadb \
    -qU langchain-chroma

In [21]:
import langchain
import langchain_groq
import pypdf
import fastembed
import chromadb
from langchain_community.document_loaders import PyPDFLoader
import os

print("All packages installed successfully!")

All packages installed successfully!


=======================================================\
pypdf : to load a pdf \
fastembde : to use the embedding \
chromdb : to store the embedding 


# 2.import the llm and the embedding model 

In [60]:
from kaggle_secrets import UserSecretsClient
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_groq import ChatGroq
import os
#llm
user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY


class Models:
    def __init__(
        self,
        Chat_model="openai/gpt-oss-20b",
        embedding_model="thenlper/gte-large"
    ):
        self.Chat_model = Chat_model
        self.embedding_model_name = embedding_model

        self.chat_llm = ChatGroq(
            model=self.Chat_model,
            temperature=0.2,
            max_tokens=2000,
        )

        self.embedding_model = FastEmbedEmbeddings(
            model_name=self.embedding_model_name
        )

# load the pdf 

In [23]:
class File_Loader:
    ALLOWED_FILES = ["pdf"]

    def __init__(self, File_links):
        self.File_links = File_links

    def get_file_type(self, file_link):
        extension = os.path.splitext(file_link)[1].lower().replace(".", "")

        if extension in self.ALLOWED_FILES:
            return extension

        return "none"

    def load_files(self):
        files = []

        for file_link in self.File_links:
            file_type = self.get_file_type(file_link)

            files.append({
                "path": file_link,
                "type": file_type
            })

        return files

    def load_allowed_files(self):
        documents = []

        files = self.load_files()

        for file in files:
            if file["type"] == "pdf":
                loader = PyPDFLoader(file["path"])
                documents.extend(loader.load())

        return documents

file_links = [
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.pdf",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.txt",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/test.docx"
]

loader = File_Loader(file_links)

documents = loader.load_allowed_files()

print(f"Loaded documents: {len(documents)}")

Loaded documents: 11


# splitting the file 

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

class Chunker:
    def __init__(self, File_links, chunk_size=1000, chunk_overlap=150):

        # Chunker internally uses File_Loader
        self.file_loader = File_Loader(File_links)

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    def split(self):
        # Load allowed files
        documents = self.file_loader.load_allowed_files()

        # Split documents into chunks
        return self.splitter.split_documents(documents)

file_links = [
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.pdf",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.txt",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/test.docx"
]

chunker = Chunker(file_links)

chunks = chunker.split()

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 27


# embedding 

In [50]:
from langchain_chroma import Chroma

class Embedding:
    def __init__(
        self,
        File_links,
        embedding_model="thenlper/gte-large",
        chat_model="openai/gpt-oss-20b",
        persist_directory="./chroma_db",
        chunk_size=800,
        chunk_overlap=150
    ):
        self.File_links = File_links
        self.persist_directory = persist_directory

        # Use LLM class
        self.Models = Models(
            Chat_model=chat_model,
            embedding_model=embedding_model
        )

        # Get embedding model from LLM
        self.embedding_model = self.Models.embedding_model

        # Get chat model from LLM
        self.chat_llm = self.Models.chat_llm

        # Chunker internally uses File_Loader
        self.chunker = Chunker(
            File_links,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        # Create chunks
        self.chunks = self.chunker.split()

        # Create ChromaDB
        self.db = Chroma.from_documents(
            documents=self.chunks,
            embedding=self.embedding_model,
            persist_directory=self.persist_directory
        )

    def get_db(self):
        return self.db

    def get_chunks(self):
        return self.chunks

    def get_Models(self):
        return self.chat_Models


In [51]:
query = "What is the importance of the title 'I Sell My Dreams ?"

# retriever part [we will use the mmr]

In [52]:
from langchain_community.retrievers import BM25Retriever


class Retriever:

    def __init__(
        self,
        embedding,
        documents,
        search_type="mmr",
        k=8,
        bm25_k=8
    ):
        self.db = embedding.get_db()

        self.k = k

        # Semantic / vector retriever
        self.retriever = self.db.as_retriever(
            search_type=search_type,
            search_kwargs={"k": k}
        )

        # BM25 retriever
        self.bm25_retriever = BM25Retriever.from_documents(
            documents
        )

        self.bm25_retriever.k = bm25_k

    def invoke(self, query):

        # Semantic search
        semantic_docs = self.retriever.invoke(query)

        # BM25 search
        bm25_docs = self.bm25_retriever.invoke(query)

        # RRF
        return self._rrf(
            semantic_docs,
            bm25_docs
        )
#Some chunks appear in both lists, so RRF combines their rankings and removes duplicates.
    def _rrf(
        self,
        semantic_docs,
        bm25_docs,
        rrf_k=60
    ):
        scores = {}
        documents = {}

        # Semantic results
        for rank, doc in enumerate(
            semantic_docs,
            start=1
        ):
            key = self._document_key(doc)

            scores[key] = scores.get(key, 0) + (
                1 / (rrf_k + rank)
            )

            documents[key] = doc

        # BM25 results
        for rank, doc in enumerate(
            bm25_docs,
            start=1
        ):
            key = self._document_key(doc)

            scores[key] = scores.get(key, 0) + (
                1 / (rrf_k + rank)
            )

            documents[key] = doc

        # Sort by RRF score
        ranked_keys = sorted(
            scores,
            key=scores.get,
            reverse=True
        )

        # Return final K documents
        return [
            documents[key]
            for key in ranked_keys[:self.k]
        ]

    def _document_key(self, doc):

        return (
            doc.page_content,
            str(sorted(doc.metadata.items()))
        )

# RAG 

In [61]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


class RAG_Retrieval:

    def __init__(self, file_links):
        # VectorDB is created ONCE
        self.embedding = Embedding(
            File_links=file_links
        )

    def get_retriever(self, k=4):
        retriever = self.embedding.get_db().as_retriever(
            search_type="mmr",
            search_kwargs={"k": k}
        )

        return retriever




class RAG_Generation:

    def __init__(
        self,
        retrieval,
        k=8,
        template="""
You are an AI assistant that answers questions using the provided context.

Rules:

- If the context does not contain enough information to answer the question,
  say exactly: "I don't know."
- if you have any information that can help say it 
- try to use all context to answer 

Context:
{context}

Question:
{query}

Answer:
"""
    ):

        self.retrieval = retrieval
        self.k = k

        self.models = Models()

        self.prompt = ChatPromptTemplate.from_template(template)

        self.retriever = self.retrieval.get_retriever(
            k=self.k
        )

        self.chain = (
            {
                "context": self.retriever,
                "query": RunnablePassthrough()
            }
            | self.prompt
            | self.models.chat_llm
            | StrOutputParser()
        )

    def response(self, query):
        return self.chain.invoke(query)


In [54]:
file_links = [
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.pdf",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.txt",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/test.docx"
]

In [55]:
rag_retrieval = RAG_Retrieval(file_links)

In [62]:
rag = RAG_Generation(
    rag_retrieval,
)

print(rag.response(
    "What is the significance of the title 'I Sell My Dreams'?"
))

**Answer:**

The title *“I Sell My Dreams”* is a metaphor for the narrator’s act of turning his personal visions and experiences into a commodity—stories that he offers to others. In the text he is not literally selling a dream in the sense of a prophetic vision, but he is “selling” the narrative of Frau Frieda’s life and her supposed prophetic dreams to the Portuguese ambassador, to the poet Neruda, and ultimately to the reader. The title therefore signals the theme that personal, inner experiences can be packaged and traded as stories, and it hints at the narrator’s own desire to monetize or share his dreams rather than keep them private.


In [66]:
import re

def norm(s):
    s = s.lower().replace("\u00a0", " ")
    s = re.sub(r"[*_`]", "", s)
    return re.sub(r"\s+", " ", s).strip()

IDK = ["i don't know", "not in", "no information", "not mentioned", "cannot"]

tests = [
    ("Who did the narrator see in Barcelona before the disaster in Havana?", "frau frieda"),
    ("Where was Neruda traveling to on his sea voyage?", "valparaiso"),
    ("How much did Neruda pay for the old book?", "two months"),
    ("What was the name of Neruda's wife?", "matilde"),
    ("Why did Matilde put a bib on Neruda?", "sauce"),
    ("How many lobsters did Neruda eat?", "three"),
    ("Which historical figure is Neruda compared to?", "renaissance pope"),
    ("What did Neruda say in a very quiet voice?", "behind him"),
    # Should be unanswerable
    ("What is the significance of the title 'I Sell My Dreams'?", None),
    ("What year does the Neruda scene take place?", None),
]



import time



passed = 0
for q, kw in tests:
    time.sleep(1.5)
    a = norm(str(rag.response(q)))
    idk = any(p in a for p in IDK)
    if kw:
        status = "PASS" if kw in a else ("MISS (idk)" if idk else "WRONG")
    else:
        status = "PASS" if idk else "HALLUCINATED"
    passed += status == "PASS"
    print(f"{status:12} | {q}\n             -> {a[:120]}\n")

print(f"Score: {passed}/{len(tests)}")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01knh881fpf1sb836hm2hm4ffh` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197516, Requested 4808. Please try again in 16m43.968s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}